1. Install Libraries

In [1]:
!pip install transformers torch scikit-learn pandas

2. Import Libraries

In [2]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

3. Load Dataset

In [3]:
df = pd.read_csv("C:/Users/srini/Downloads/IMDB Dataset.csv")
print(df.head())
print(df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


In [4]:
df = df.sample(5000)

4. Data Preprocessing

In [5]:
# Remove missing values
df.dropna(inplace=True)

# Encode labels (positive = 1, negative = 0)
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['sentiment'])

# Input and labels
texts = df['review'].tolist()
labels = df['label'].tolist()

5. Train / Validation / Test Split

In [6]:
X_train, X_temp, y_train, y_temp = train_test_split(texts, labels, test_size=0.2, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

6. Tokenization


In [7]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=128)

7. Custom Dataset Class

In [8]:
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbDataset(train_encodings, y_train)
val_dataset = IMDbDataset(val_encodings, y_val)
test_dataset = IMDbDataset(test_encodings, y_test)

8. Load BERT Model

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


9. Training Arguments

In [22]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


10. Metrics Function

In [23]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

11. Trainer

In [25]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)
trainer.remove_callback(type(trainer.callback_handler.callbacks[-1]))

12. Train Model

C:\Users\srini\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.40029653906822205, 'eval_model_preparation_time': 0.0692, 'eval_accuracy': 0.882, 'eval_precision': 0.8843283582089553, 'eval_recall': 0.8943396226415095, 'eval_f1': 0.8893058161350844, 'eval_runtime': 67.2024, 'eval_samples_per_second': 7.44, 'eval_steps_per_second': 0.937, 'epoch': 0}


In [16]:
trainer.train()

C:\Users\srini\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.422044,0.354443,0.860000,0.929515,0.796226,0.857724
2,0.232951,0.400297,0.882000,0.884328,0.894340,0.889306


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\srini\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.3274973831176758, metrics={'train_runtime': 5502.1866, 'train_samples_per_second': 1.454, 'train_steps_per_second': 0.182, 'total_flos': 526222110720000.0, 'train_loss': 0.3274973831176758, 'epoch': 2.0})

13. Evaluate

In [27]:
results = trainer.evaluate()
print(results)

C:\Users\srini\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.40029653906822205, 'eval_model_preparation_time': 0.0692, 'eval_accuracy': 0.882, 'eval_precision': 0.8843283582089553, 'eval_recall': 0.8943396226415095, 'eval_f1': 0.8893058161350844, 'eval_runtime': 62.3625, 'eval_samples_per_second': 8.018, 'eval_steps_per_second': 1.01, 'epoch': 0}


14. Final Output

In [28]:
print("Running final predictions...")

predictions = trainer.predict(test_dataset)

y_pred = predictions.predictions.argmax(axis=1)

print("\nFinal Results:\n")

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Running final predictions...


C:\Users\srini\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Final Results:

Accuracy: 0.856
Precision: 0.8524590163934426
Recall: 0.8524590163934426
F1 Score: 0.8524590163934426

Confusion Matrix:
 [[220  36]
 [ 36 208]]


Freeze BERT

In [31]:
for param in model.bert.parameters():
    param.requires_grad = False

Fine-tune Last 2 Layers

In [32]:
for name, param in model.bert.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False